In [ ]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

import gc, math, torch, time
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# IDs
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"             # base (fp16) for reference
finetuned_repo_id = "eduhuemar001/tinyllama-german-checkpoints-sentiment-4bit"  # your fine-tuned repo

# Helpers
def print_gpu_mem(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA not available"); return 0
    torch.cuda.synchronize()
    a = torch.cuda.memory_allocated()
    print(f"[{tag}] allocated={a/1e9:.3f} GB"); return a

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

# ---------- fp16 base (reference) ----------
free_gpu()
print_gpu_mem("before fp16")
base_fp16 = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="cuda" if torch.cuda.is_available() else None,
)
base_fp16.eval()
alloc_fp16 = print_gpu_mem("after  fp16")

# cleanup fp16
del base_fp16
free_gpu()
print_gpu_mem("after unload fp16")

# ---------- 4-bit: load the *fine-tuned* model ----------
# compute dtype: bf16 on Ampere+, else fp16
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print_gpu_mem("before 4-bit finetuned")

model_4bit = None
try:
    # Case A: repo contains LoRA adapters -> attach to 4-bit base
    from peft import PeftModel
    base_4bit = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=quant_config,
        device_map="auto",
    )
    model_4bit = PeftModel.from_pretrained(base_4bit, finetuned_repo_id)  # adapter_model.safetensors
except Exception:
    # Case B: repo is a merged fine-tuned model -> load directly in 4-bit
    model_4bit = AutoModelForCausalLM.from_pretrained(
        finetuned_repo_id,
        quantization_config=quant_config,
        device_map="auto",
    )

model_4bit.eval()
model_4bit.config.use_cache = True
alloc_4bit = print_gpu_mem("after  4-bit finetuned")

print("\n=== GPU allocated deltas ===")
print(f"fp16 base     ≈ {alloc_fp16/1e9:.3f} GB")
print(f"4-bit finetuned ≈ {alloc_4bit/1e9:.3f} GB")